### 1. Clasificación de Elementos del Dominio

#### Entities (Entidades)
Poseen una identidad única y continua a lo largo del tiempo, independientemente de que sus atributos cambien. Tienen un ciclo de vida propio.

* **Usuario:** Identidad global definida por su `username`. Rastrea el estado de visualización, listas personales y configuración de suscripción.
* **Serie:** Identidad global definida por su `id` alfanumérico. Es la raíz del catálogo y agrupa la jerarquía de contenido.
* **Temporada:** Entidad con identidad local. Su existencia y numeración solo tienen sentido dentro del contexto de una Serie concreta.
* **Capítulo:** Entidad con identidad local. Unidad mínima de contenido, dependiente estructuralmente de una Temporada.
* **Factura:** Identidad global definida por un código de facturación único. Representa una obligación de pago estática en un periodo de tiempo.
* **Persona:** Identidad global definida por un `id` o `nombre`. Representa actores y creadores de forma centralizada para evitar duplicidades cuando participan en múltiples series o asumen distintos roles.

#### Value Objects (Objetos de Valor)
Representan atributos conceptuales o medidas. Carecen de identidad propia y deben ser inmutables. Si sus datos cambian, se reemplaza el objeto entero.

* **IBAN:** Encapsula la validación y el formato de una cuenta bancaria.
* **PlanSuscripcion:** Define el modelo de negocio (Tarifa Plana vs. Pago por Visión) y la cuota, sin identidad intrínseca.
* **LineaFactura:** Describe un cargo inmutable en el tiempo (fecha, concepto, importe base). No tiene sentido fuera de la factura que la contiene.

---

### 2. Aggregates y Aggregate Roots

#### Aggregate: Usuario
* **Aggregate Root:** `Usuario`
* **Justificación:** Es el núcleo transaccional de la actividad del cliente. El Root garantiza que las transiciones de estado sean coherentes. Por ejemplo, asegura que al añadir un capítulo al historial, el "último capítulo visto" se actualice correctamente siguiendo el orden lógico de las temporadas, y sincroniza el estado general de la serie (EMPEZADA, TERMINADA). Mantiene referencias a otros Aggregates (Series) exclusivamente a través de sus identificadores para evitar un acoplamiento profundo.

#### Aggregate: Serie
* **Aggregate Root:** `Serie`
* **Elementos internos:** `Temporada`, `Capítulo`
* **Justificación:** Define la frontera de consistencia del catálogo de contenido. Un capítulo o temporada no tiene ciclo de vida fuera de su serie. El Root (`Serie`) protege la integridad de la estructura jerárquica y encapsula el acceso a la información profunda, garantizando que no existan capítulos huérfanos.

#### Aggregate: Facturación
* **Aggregate Root:** `Factura`
* **Elementos internos:** `LineaFactura`
* **Justificación:** Ejerce como unidad de trabajo inmutable para los cobros. Protege la invariante económica del sistema: garantiza que el importe total siempre refleje la suma exacta de sus líneas de cargo y evita modificaciones en periodos fiscales ya cerrados. Centraliza la adición de cargos en un punto único.

#### Aggregate: Persona
* **Aggregate Root:** `Persona`
* **Justificación:** Al ser actores y creadores compartidos por múltiples series en la plataforma, requieren un ciclo de vida independiente. Son referenciados por el Aggregate de Serie, pero no pertenecen a él.

---

### 3. Justificaciones de Diseño (Reglas DDD)

#### Referencias por Identidad entre Aggregates
En estricto cumplimiento con las normativas de DDD, los Aggregates distintos no mantienen referencias directas en memoria mediante objetos complejos bidireccionales, sino a través de identificadores de valor (IDs). Por ejemplo, el Usuario almacena en sus mapas el ID de la Serie (`String`), no la entidad completa. Esto reduce el acoplamiento estructural, previene la carga indiscriminada de grafos de objetos masivos por parte del ORM (JPA/Hibernate) y respeta las fronteras de las transacciones.

#### Protección de Invariantes y Lógica de Dominio Centralizada
El diseño fomenta el patrón de "Modelo de Dominio Rico" (Rich Domain Model). En lugar de tener servicios anémicos que modifican getters y setters de entidades vacías, las reglas de negocio críticas residen en las propias entidades. Por ejemplo, el método de registrar una visualización reside dentro de Usuario, siendo este el único responsable de orquestar la actualización de su historial, el cambio de estado de la serie y la orden de generar un cargo en la Factura.